# Visualization of surfaces
Running requires left and right surfaces, a dataset instance, a gifti or cifti label file, and specification of centroids. 
We show single instances of how these can be run after a full run of CPM 


In [1]:
import json
import numpy as np
import nibabel as nib
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.spatial.distance import cdist
from surfdist.utils import AnatomyInputParser,recort
from surfdist.analysis import shortest_path_pygeodesic,calc_roi_dist,dist_calc
from surfdist.load import load_cifti_labels,load_gifti_labels
from brain2behaviour.dataset import BrainBehaviorDataset

The following two functions do all the work. The rest just shows how we ran things. 
We cannot share dataset objects themselves as they expose confidential data. However you can recreate a run like this. 

In [2]:
def save_dict_as_gifti(data_dict, output_path):
    """
    Save a dictionary as a GIFTI (.gii) file.

    Each key in the dictionary is stored as a label in metadata,
    and each value (array-like) is stored as a separate DataArray.

    Parameters
    ----------
    data_dict : dict
        Dictionary where keys are strings (labels) and values are array-like (1D or 2D).
    output_path : str
        Path to save the output .gii file
    """
    data_arrays = []
    for label, arr in data_dict.items():
        arr = np.asarray(arr, dtype=np.float32)
        gda = nib.gifti.GiftiDataArray(data=arr)
        gda.meta['Name'] = label   # modern API: treat meta like dict
        data_arrays.append(gda)

    img = nib.gifti.GiftiImage(darrays=data_arrays)
    nib.save(img, output_path)
    print(f"GIFTI file saved to {output_path}")

In [3]:
def load_dict_from_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
        

def get_edge_path(edge, surf, cortex, labels):
    edge = edge.split('--')
    vtx_coords, _ = AnatomyInputParser(surf)

    # --- Source ---
    source_nodes = np.atleast_1d(labels[edge[0]])
    source_coords = vtx_coords[source_nodes]
    source_centroid_crd = np.mean(source_coords, axis=0)
    dists_ctr_source = cdist(source_coords, source_centroid_crd[None, :], metric='euclidean')
    source_node = source_nodes[np.argmin(dists_ctr_source)]

    # --- Target ---
    target_nodes = np.atleast_1d(labels[edge[1]])
    target_coords = vtx_coords[target_nodes]
    target_centroid_crd = np.mean(target_coords, axis=0)
    dists_ctr_target = cdist(target_coords, target_centroid_crd[None, :], metric='euclidean')
    target_node = target_nodes[np.argmin(dists_ctr_target)]

    # --- Geodesic path ---
    geopath = shortest_path_pygeodesic(surf, cortex, source_node, target_node)
    return geopath




In [ ]:
def plot_paths_on_cortex(left_surf,right_surf,dataset,labels,task,lh_prefix,rh_prefix,centroid=False):
    """Plotting functoin to get paths onto a cortical surface. -- for visualization only. 
    if using gifti labels then pass a list like so [lh_labels, rh_labels]
    otherwise pass a single cifti file path
    """
    if type(labels)==list:
        print('using gifti labels')
        lh_labels=load_gifti_labels(labels[0])        
        rh_labels=load_gifti_labels(labels[1])
    else:
        print('using cifti labels')
        lh_labels=load_cifti_labels(labels,'L')
        rh_labels=load_cifti_labels(labels,'R')

  
    ##### get the medial wall and cortex masks 
    L_medial_wall=lh_labels['???']
    Ldims=len(np.hstack(list(lh_labels.values())))
    Lcort = np.arange(Ldims)

    Lcort=np.delete(Lcort,L_medial_wall)

    R_medial_wall=rh_labels['???']
    Rdims=len(np.hstack(list(rh_labels.values())))
    Rcort = np.arange(Rdims)
    Rcort=np.delete(Rcort,R_medial_wall)

    #### get the features
    print(dataset.filepath)
    features_path=dataset.filepath.replace('.pkl','_cpm_analysis/')+'ds_permed0000_stable.json'
    feats=load_dict_from_json(features_path)
    dataset.features=feats
    pos_feats=dataset.features[task]['positive']
    print(f'there are {len(pos_feats)} positive features')
    neg_feats=dataset.features[task]['negative']
    print(f'there are {len(neg_feats)} negative features')


    ### separate features by hemisphere and sign
    lh_neg=[i for i in neg_feats if lh_prefix in i]
    if len(lh_neg)>0:
        if '.lh' in lh_neg[0]:
            lh_neg=[i.replace('.lh','') for i in lh_neg if '.lh' in i]
    
    rh_neg=[i for i in neg_feats if rh_prefix in i]  
    if len(rh_neg)>0:
        if '.rh' in rh_neg[0]:
            rh_neg=[i.replace('.rh','') for i in rh_neg if '.rh' in i]

    lh_pos=[i for i in pos_feats if lh_prefix in i]
    if len(lh_pos)>0:
        if '.lh' in lh_pos[0]:
            lh_pos=[i.replace('.lh','') for i in lh_pos if '.lh' in i]

    rh_pos=[i for i in pos_feats if rh_prefix in i]
    if len(rh_pos)>0:
        if '.rh' in rh_pos[0]:
            rh_pos=[i.replace('.rh','') for i in rh_pos if '.rh' in i]
    
    #### left hemi 
    lh_neg_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, left_surf, Lcort, lh_labels) 
    for i in lh_neg)
    lh_neg_vertices = np.unique(np.hstack(lh_neg_paths))
    
    lh_pos_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, left_surf, Lcort, lh_labels) 
    for i in lh_pos)
    lh_pos_vertices = np.unique(np.hstack(lh_pos_paths))
    
    ### right hemi 
    rh_neg_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, right_surf, Rcort, rh_labels) 
    for i in rh_neg)
    rh_neg_vertices = np.unique(np.hstack(rh_neg_paths))
    
    rh_pos_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, right_surf, Rcort, rh_labels) 
    for i in rh_pos)
    rh_pos_vertices = np.unique(np.hstack(rh_pos_paths))

    lh_neg=np.zeros(Ldims)
    lh_neg[lh_neg_vertices]=1
    lh_pos=np.zeros(Ldims)
    lh_pos[lh_pos_vertices]=1

    rh_neg=np.zeros(Rdims)
    rh_neg[rh_neg_vertices]=1
    rh_pos=np.zeros(Rdims)
    rh_pos[rh_pos_vertices]=1
    

    return lh_neg,lh_pos,rh_neg,rh_pos

In [ ]:
def return_path_dict(left_surf,right_surf,dataset,labels,task,lh_prefix,rh_prefix,centroid=False):
    """Plotting functoin to get paths onto a cortical surface. -- for visualization only. 
    if using gifti labels then pass a list like so [lh_labels, rh_labels]
    otherwise pass a single cifti file path
    """
    if type(labels)==list:
        print('using gifti labels')
        lh_labels=load_gifti_labels(labels[0])        
        rh_labels=load_gifti_labels(labels[1])
    else:
        print('using cifti labels')
        lh_labels=load_cifti_labels(labels,'L')
        rh_labels=load_cifti_labels(labels,'R')

  
    ##### get the medial wall and cortex masks 
    L_medial_wall=lh_labels['???']
    Ldims=len(np.hstack(list(lh_labels.values())))
    Lcort = np.arange(Ldims)

    Lcort=np.delete(Lcort,L_medial_wall)

    R_medial_wall=rh_labels['???']
    Rdims=len(np.hstack(list(rh_labels.values())))
    Rcort = np.arange(Rdims)
    Rcort=np.delete(Rcort,R_medial_wall)

    #### get the features
    print(dataset.filepath)
    features_path=dataset.filepath.replace('.pkl','_cpm_analysis/')+'ds_permed0000_stable.json'
    feats=load_dict_from_json(features_path)
    dataset.features=feats
    pos_feats=dataset.features[task]['positive']
    print(f'there are {len(pos_feats)} positive features')
    neg_feats=dataset.features[task]['negative']
    print(f'there are {len(neg_feats)} negative features')


    ### separate features by hemisphere and sign
    lh_neg=[i for i in neg_feats if lh_prefix in i]
    if len(lh_neg)>0:
        if '.lh' in lh_neg[0]:
            lh_neg=[i.replace('.lh','') for i in lh_neg if '.lh' in i]
        lh_neg_names=lh_neg.copy()
    
    rh_neg=[i for i in neg_feats if rh_prefix in i]  
    if len(rh_neg)>0:
        if '.rh' in rh_neg[0]:
            rh_neg=[i.replace('.rh','') for i in rh_neg if '.rh' in i]
        rh_neg_names=rh_neg.copy()

    lh_pos=[i for i in pos_feats if lh_prefix in i]
    if len(lh_pos)>0:
        if '.lh' in lh_pos[0]:
            lh_pos=[i.replace('.lh','') for i in lh_pos if '.lh' in i]
        lh_pos_names=lh_pos.copy()

    rh_pos=[i for i in pos_feats if rh_prefix in i]
    if len(rh_pos)>0:
        if '.rh' in rh_pos[0]:
            rh_pos=[i.replace('.rh','') for i in rh_pos if '.rh' in i]
        rh_pos_names=rh_pos.copy()
    
    #### left hemi 
    lh_neg_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, left_surf, Lcort, lh_labels) 
    for i in lh_neg)
    lh_neg_vertices = np.unique(np.hstack(lh_neg_paths))
    
    lh_pos_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, left_surf, Lcort, lh_labels) 
    for i in lh_pos)
    lh_pos_vertices = np.unique(np.hstack(lh_pos_paths))
    
    ### right hemi 
    rh_neg_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, right_surf, Rcort, rh_labels) 
    for i in rh_neg)
    rh_neg_vertices = np.unique(np.hstack(rh_neg_paths))
    
    rh_pos_paths = Parallel(n_jobs=-1)(  # -1 = use all cores
    delayed(get_edge_path)(i, right_surf, Rcort, rh_labels) 
    for i in rh_pos)
    rh_pos_vertices = np.unique(np.hstack(rh_pos_paths))
    lh_neg_paths=dict(zip(lh_neg_names,lh_neg_paths))
    lh_pos_paths=dict(zip(lh_pos_names,lh_pos_paths))
    rh_neg_paths=dict(zip(rh_neg_names,rh_neg_paths))
    rh_pos_paths=dict(zip(rh_pos_names,rh_pos_paths))

    
    
                

    return lh_neg_paths,lh_pos_paths,rh_neg_paths,rh_pos_paths

In [21]:
l_anat='brain2behaviour/atlases/StandardSurface32k_fs_LR/midInflated.L.surf.gii'
r_anat='brain2behaviour/atlases/StandardSurface32k_fs_LR/midInflated.R.surf.gii'

Example how to get feature plots 

In [ ]:
dist_neg_lh={}
dist_neg_rh={}
dist_pos_lh={}
dist_pos_rh={}
schaefer_dims=[400]
for dim in schaefer_dims:
    ds=BrainBehaviorDataset.load(f'<Path to schaefer 400 dataset.pkl>')
    atlas=f'brain2behaviour/atlases/schaefer/Schaefer2018_{dim}Parcels_7Networks_order.dlabel.nii'
    lh_neg,lh_pos,rh_neg,rh_pos=return_path_dict(l_anat,r_anat,ds,atlas,'ReadEng_Unadj','LH','RH')
    dist_neg_lh[f'Schaefer {dim}']=lh_neg
    dist_pos_lh[f'Schaefer {dim}']=lh_pos

    dist_neg_rh[f'Schaefer {dim}']=rh_neg
    dist_pos_rh[f'Schaefer {dim}']=rh_pos


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


using cifti labels


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


Running on a Non HCP CIFTI
Running on a Non HCP CIFTI
/well/margulies/projects/DistanceValidation/data/HCP//Distance/Schaefer400/Reading_SES_DistanceCentroid.pkl
there are 234 positive features
there are 153 negative features


In [27]:
lh_neg = {k.replace('7Networks_LH_', ''): v for k, v in lh_neg.items()}
lh_pos = {k.replace('7Networks_LH_', ''): v for k, v in lh_pos.items()}

In [29]:
rh_neg = {k.replace('7Networks_RH_', ''): v for k, v in rh_neg.items()}
rh_pos = {k.replace('7Networks_RH_', ''): v for k, v in rh_pos.items()}

In [38]:
for i in lh_neg:
    out=np.zeros(32492)
    out[lh_neg[i]]=1
    lh_neg[i]=out

for i in lh_pos:
    out=np.zeros(32492)
    out[lh_pos[i]]=1
    lh_pos[i]=out

for i in rh_neg:
    out=np.zeros(32492)
    out[rh_neg[i]]=1
    rh_neg[i]=out

for i in rh_pos:
    out=np.zeros(32492)
    out[rh_pos[i]]=1
    rh_pos[i]=out

In [ ]:
### get individual paths 

In [40]:
save_dict_as_gifti(lh_neg,'gifti_results/Schaefer400.L.Indi.Neg.Paths.func.gii')
save_dict_as_gifti(lh_pos,'gifti_results/Schaefer400.L.Indi.Pos.Paths.func.gii')

save_dict_as_gifti(rh_neg,'gifti_results/Schaefer400.R.Indi.Neg.Paths.func.gii')
save_dict_as_gifti(rh_pos,'gifti_results/Schaefer400.R.Indi.Pos.Paths.func.gii')

GIFTI file saved to gifti_results/Schaefer400.L.Indi.Neg.Paths.func.gii
GIFTI file saved to gifti_results/Schaefer400.L.Indi.Pos.Paths.func.gii
GIFTI file saved to gifti_results/Schaefer400.R.Indi.Neg.Paths.func.gii
GIFTI file saved to gifti_results/Schaefer400.R.Indi.Pos.Paths.func.gii


In [ ]:
save_dict_as_gifti(FC_neg_lh,'gifti_results/YA.FC.L.Neg.func.gii')
save_dict_as_gifti(FC_pos_lh,'gifti_results/YA.FC.L.Pos.func.gii')

save_dict_as_gifti(FC_neg_rh,'gifti_results/YA.FC.R.Neg.func.gii')
save_dict_as_gifti(FC_pos_rh,'gifti_results/YA.FC.R.Pos.func.gii')

In [ ]:
def plot_suraceareas(left_surf,right_surf,dataset,labels,task,lh_prefix,rh_prefix):
    """Plotting functioin to get paths onto a cortical surface. -- for visualization only. 
    if using gifti labels then pass a list like so [lh_labels, rh_labels]
    otherwise pass a single cifti file path
    """
    if type(labels)==list:
        print('using gifti labels')
        lh_labels=load_gifti_labels(labels[0])        
        rh_labels=load_gifti_labels(labels[1])
    else:
        print('using cifti labels')
        lh_labels=load_cifti_labels(labels,'L')
        rh_labels=load_cifti_labels(labels,'R')

  
    ##### get the medial wall and cortex masks 
    L_medial_wall=lh_labels['???']
    Ldims=len(np.hstack(list(lh_labels.values())))
    Lcort = np.arange(Ldims)

    Lcort=np.delete(Lcort,L_medial_wall)

    R_medial_wall=rh_labels['???']
    Rdims=len(np.hstack(list(rh_labels.values())))
    Rcort = np.arange(Rdims)
    Rcort=np.delete(Rcort,R_medial_wall)

    ##### get the features
    print(dataset.filepath)
    features_path=dataset.filepath.replace('.pkl','_cpm_analysis/')+'ds_permed0000_stable.json'
    feats=load_dict_from_json(features_path)
    dataset.features=feats

    pos_feats=dataset.features[task]['positive']
    print(f'there are {len(pos_feats)} positive features')
    neg_feats=dataset.features[task]['negative']
    print(f'there are {len(neg_feats)} negative features')


     ### separate features by hemisphere and sign
    lh_neg=[i for i in neg_feats if lh_prefix in i]
    if len(lh_neg)>0:
        if '.lh' in lh_neg[0]:
            lh_neg=[i.replace('.lh','') for i in lh_neg if '.lh' in i]
    
    rh_neg=[i for i in neg_feats if rh_prefix in i]  
    if len(rh_neg)>0:
        if '.rh' in rh_neg[0]:
            rh_neg=[i.replace('.rh','') for i in rh_neg if '.rh' in i]

    lh_pos=[i for i in pos_feats if lh_prefix in i]
    if len(lh_pos)>0:
        if '.lh' in lh_pos[0]:
            lh_pos=[i.replace('.lh','') for i in lh_pos if '.lh' in i]

    rh_pos=[i for i in pos_feats if rh_prefix in i]
    if len(rh_pos)>0:
        if '.rh' in rh_pos[0]:
            rh_pos=[i.replace('.rh','') for i in rh_pos if '.rh' in i]


    lh_vect=np.zeros(32492)
    for i in lh_neg:
        roi=lh_labels[i]
        lh_vect[roi]=-1
    for i in lh_pos:
        lh_vect[lh_labels[i]]=1

    rh_vect=np.zeros(32492)
    for i in rh_neg:
        rh_vect[rh_labels[i]]=-1
    for i in rh_pos:
        rh_vect[rh_labels[i]]=1
    return lh_vect,rh_vect

In [ ]:
SA_lh_data={}
SA_rh_data={}
schaefer_dims=[200,400,600,800,1000]
for dim in schaefer_dims:
    ds=BrainBehaviorDataset.load(f'<path>/SurfaceArea/Schaefer{dim}/Reading_SES_SurfaceArea.pkl')
    atlas=f'brain2behaviour/atlases/schaefer/Schaefer2018_{dim}Parcels_7Networks_order.dlabel.nii'
    lh_sa,rh_sa=plot_suraceareas(l_anat,r_anat,ds,atlas,'ReadEng_Unadj','LH','RH')
    SA_lh_data[f'Schaefer {dim}']=lh_sa
    SA_rh_data[f'Schaefer {dim}']=rh_sa

In [ ]:
save_dict_as_gifti(SA_lh_data,'gifti_results/YA.SA.L.func.gii')
save_dict_as_gifti(SA_rh_data,'gifti_results/YA.SA.R.func.gii')

